<a href="https://colab.research.google.com/github/evkoff/DI-Bootcamp-Stage1/blob/main/Week14/Day4/ExerciseXP/W14D4__XP_VDB_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [4]:
# Install compatible packages for modern Google Colab

%pip install -q \
    faiss-cpu \
    chromadb \
    sentence-transformers \
    transformers



In [12]:
!pip uninstall -y chromadb

Found existing installation: chromadb 0.3.21
Uninstalling chromadb-0.3.21:
  Successfully uninstalled chromadb-0.3.21


In [13]:
!pip show chromadb

In [14]:
!pip install -U chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

In [15]:
!pip show chromadb

Name: chromadb
Version: 1.5.9
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 


In [16]:
import chromadb
print(chromadb.__version__)

1.5.9


In [18]:
from chromadb.config import Settings

In [19]:
import pydantic
print(pydantic.__version__)

2.13.4


In [20]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer, InputExample

import chromadb
from chromadb.config import Settings

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from IPython.display import display

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

## 🌟 Exercise 1 · Data loading and preparation

In [24]:
data_path = 'labelled_newscatcher_dataset.csv'
pdf = pd.read_csv(data_path, sep=';')
if "id" not in pdf.columns:
    pdf["id"] = range(len(pdf))  # TODO: replace with your own identifier logic if provided
display(pdf.head())
# TODO: create a manageable subset (e.g., first 1000 rows)
pdf_subset = pdf.head(1000)
pdf_subset[['id', 'title']].head()


,topic,link,domain,published_date,title,lang,id
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en,0
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en,1
2,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en,2
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en,3
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en,4


,id,title
0,0,A closer look at water-splitting's solar fuel ...
1,1,"An irresistible scent makes locusts swarm, stu..."
2,2,Artificial intelligence warning: AI will know ...
3,3,Glaciers Could Have Sculpted Mars Valleys: Study
4,4,Perseid meteor shower 2020: What time and how ...


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [25]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# Todo: create training examples from the subset data using the example_create_fn

faiss_train_examples = [
    example_create_fn(row.id, row.title)
    for row in pdf_subset.itertuples()
]
faiss_train_examples[:2]


In [26]:
example = faiss_train_examples[0]

print(example.guid)
print(example.texts)
print(example.label)

0
["A closer look at water-splitting's solar fuel potential"]
0.0


In [27]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(
    titles_list,
    convert_to_numpy=True,
    show_progress_bar=True
)
len(faiss_title_embedding), len(faiss_title_embedding[0])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(1000, 384)

## 🌟 Exercise 3 · FAISS indexing and search

In [28]:
# Use the prepared subset for indexing
pdf_to_index = pdf_subset

# Convert document IDs to a NumPy array with int64 type (required by FAISS)
id_index = pdf_to_index["id"].to_numpy().astype(np.int64)

# Convert embeddings to float32 (FAISS works with float32 vectors)
content_encoded_normalized = faiss_title_embedding.astype("float32")

# Normalize vectors to unit length for cosine similarity search
faiss.normalize_L2(content_encoded_normalized)

# Create a flat FAISS index using Inner Product similarity (IP - Inner Product - скалярное произведение. После нормализации оно эквивалентно cosine similarity.)
# The embedding dimension is inferred automatically from the data
index_content = faiss.IndexIDMap(
    faiss.IndexFlatIP(content_encoded_normalized.shape[1])
)

# Add embeddings together with their document IDs
index_content.add_with_ids(content_encoded_normalized, id_index)

# Display the total number of indexed vectors
index_content.ntotal


1000

In [29]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # Encode the query into an embedding vector
    query_vector = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize the query vector for cosine similarity search
    faiss.normalize_L2(query_vector)

    # Search the FAISS index and return top-k matches
    sims, ids = index_content.search(query_vector, k)

    # Retrieve matching documents from the DataFrame
    results = pdf_to_index[pdf_to_index["id"].isin(ids[0])].copy()

    # Attach similarity scores to the results
    results["similarities"] = sims[0]

    return results


display(search_content("animal", pdf_to_index, k=5))

,topic,link,domain,published_date,title,lang,id,similarities
99,TECHNOLOGY,https://www.gematsu.com/2020/08/ghostwire-toky...,gematsu.com,2020-08-07 16:43:13,Ghostwire: Tokyo confirms dog petting,en,99,0.391902
176,TECHNOLOGY,https://www.pushsquare.com/news/2020/08/random...,pushsquare.com,2020-08-03 16:30:00,Random: You Can Pick Up and Pet Cats in Assass...,en,176,0.376784
762,SCIENCE,https://af.reuters.com/article/worldNews/idAFK...,af.reuters.com,2020-08-13 16:51:00,'Secret' life of sharks: Study reveals their s...,en,762,0.344059
928,SCIENCE,https://www.thecut.com/2020/08/scientists-say-...,thecut.com,2020-08-04 12:52:00,Just Let This Lizard Be a Dinosaur,en,928,0.317387
975,HEALTH,https://www.news-medical.net/news/20200813/Res...,news-medical.net,2020-08-13 05:18:00,Researchers explore social behavior of animals...,en,975,0.295497


## 🌟 Exercise 4 · ChromaDB collection and querying

In [30]:
# Create a persistent ChromaDB client (data is stored on disk)
chroma_client = chromadb.PersistentClient(path="cache/chroma")

# Define the collection name
collection_name = "my_news"

# Remove the existing collection to avoid duplicate data
if collection_name in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(name=collection_name)

# To-Do: create the collection and add documents
# Create a new collection for storing news articles
collection = chroma_client.create_collection(
    name=collection_name
)
# Add documents, embeddings, IDs and metadata to the collection
collection.add(
    ids=pdf_subset["id"].astype(str).tolist(),
    documents=pdf_subset["title"].tolist(),
    embeddings=faiss_title_embedding.tolist(),
    metadatas=pdf_subset[["topic", "domain"]].to_dict("records"),
)
# Search the collection using a text query
results = collection.query(
    query_texts=["animal"],
    n_results=5
)
# Pretty-print the search results
print(json.dumps(results, indent=2))


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 31.6MiB/s]


{
  "ids": [
    [
      "176",
      "975",
      "99",
      "928",
      "762"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "Random: You Can Pick Up and Pet Cats in Assassin's Creed Valhalla",
      "Researchers explore social behavior of animals toward emerging infectious diseases",
      "Ghostwire: Tokyo confirms dog petting",
      "Just Let This Lizard Be a Dinosaur",
      "'Secret' life of sharks: Study reveals their surprising social networks"
    ]
  ],
  "uris": null,
  "included": [
    "metadatas",
    "documents",
    "distances"
  ],
  "data": null,
  "metadatas": [
    [
      {
        "domain": "pushsquare.com",
        "topic": "TECHNOLOGY"
      },
      {
        "topic": "HEALTH",
        "domain": "news-medical.net"
      },
      {
        "domain": "gematsu.com",
        "topic": "TECHNOLOGY"
      },
      {
        "domain": "thecut.com",
        "topic": "SCIENCE"
      },
      {
        "topic": "SCIENCE",
        "domain": "af.reuters.c

## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [32]:
import transformers
print(transformers.__version__)

5.14.1


In [34]:
from transformers.pipelines import SUPPORTED_TASKS

print("text2text-generation" in SUPPORTED_TASKS)
print(sorted(SUPPORTED_TASKS.keys()))

False
['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'object-detection', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']


In [36]:
# Load tokenizer and model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_id)
qa_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Define the user's question
question = "What's the latest news on space development?"

# Retrieve the top 3 documents from the search results
context_docs = results["documents"][0][:3]

# Combine retrieved documents into one context
context = " ".join(context_docs)

# Build the prompt
prompt = (
    f"Answer the question using only the context.\n\n"
    f"Context:\n{context}\n\n"
    f"Question:\n{question}\n\n"
    f"Answer:"
)

# Tokenize the prompt
inputs = tokenizer(prompt, return_tensors="pt")

# Generate an answer
outputs = qa_model.generate(**inputs, max_new_tokens=64)

# Decode the generated tokens into text
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Ghostwire: Tokyo confirms dog petting
